In [1]:
import numpy as np|
import pandas as pd

In [17]:
import yfinance as yf

In [ ]:
ticker = "RELIANCE.NS"

result = yf.download(
    ticker,
    period="2y",
    interval="1d",
    auto_adjust=False
)

result.head()

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,,
2024-09-30,1463.932495,1476.574951,1524.974976,1474.400024,1519.400024,27008814
2024-10-01,1452.283081,1464.824951,1487.949951,1462.824951,1480.650024,16431642
2024-10-03,1394.928467,1406.974976,1449.400024,1402.074951,1436.574951,37757294
2024-10-04,1374.653564,1386.525024,1417.349976,1381.775024,1403.925049,37072876
2024-10-07,1358.988892,1370.724976,1396.500000,1361.375000,1393.500000,22181340


In [21]:
df = result

In [27]:
def calculate_technical_indicators(df):

    df["Daily_Return"] = df["Close"].pct_change()
    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain/avg_loss
    df["RSI"] = 100 - (100/(1+rs))
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()

    df["MCAD"]=ema12 - ema26
    df["MCAD_Signal"] = df["MCAD"].ewm(span=9, adjust=False).mean()

    high_low = df["High"] - df["Low"]
    high_close = abs(df["High"] - df["Close"].shift())
    low_close = abs(df["Low"] - df["Close"].shift())

    true_range = pd.concat(
        [high_low, high_close, low_close],
        axis=1
    ).max(axis=1)

    df["ATR"] = true_range.rolling(14).mean()

    rolling_mean = df["Close"].rolling(20).mean()
    rolling_std = df["Close"].rolling(20).std()

    df["BB_upper"] = rolling_mean + 2 * rolling_std
    df["BB_lower"] = rolling_mean - 2 * rolling_std

    df["Volume_MA20"] = df["Volume"].rolling(20).mean()
    df["Volume_Ratio"] = df["Volume"] / df["Volume_MA20"]
    

In [28]:
def technical_agent(df):

    df = calculate_technical_indicators(df)

    latest = df.iloc[-1]

    signals = {}

    # Trend
    if latest["Close"] > latest["MA_50"]:
        signals["MA50"] = "above"
    else:
        signals["MA50"] = "below"

    if latest["Close"] > latest["MA_200"]:
        signals["MA200"] = "above"
    else:
        signals["MA200"] = "below"

    # RSI
    if latest["RSI"] > 70:
        signals["RSI"] = "overbought"
    elif latest["RSI"] < 30:
        signals["RSI"] = "oversold"
    else:
        signals["RSI"] = "neutral"

    # MACD
    if latest["MACD"] > latest["MACD_Signal"]:
        signals["MACD"] = "bullish"
    else:
        signals["MACD"] = "bearish"

    # Volume
    if latest["Volume_Ratio"] > 1.5:
        signals["Volume"] = "high"
    elif latest["Volume_Ratio"] < 0.7:
        signals["Volume"] = "low"
    else:
        signals["Volume"] = "normal"

    # Bollinger
    if latest["Close"] > latest["BB_upper"]:
        signals["Bollinger"] = "above_upper_band"
    elif latest["Close"] < latest["BB_lower"]:
        signals["Bollinger"] = "below_lower_band"
    else:
        signals["Bollinger"] = "inside_bands"

    return signals

In [29]:
print(technical_agent(df))

ValueError: Cannot set a DataFrame with multiple columns to the single column Volume_Ratio